<img src="http://developer.download.nvidia.com/notebooks/dlsw-notebooks/rivaasr-finetuning-conformer-ctc-nemo/nvidia_logo.png" style="width: 90px; float: right;">

# Nemotron 3.5 ASR Streaming — 한국어 파인튜닝 (Production)

**목표**: `nvidia/nemotron-3.5-asr-streaming-0.6b` 모델을 한국어 음성인식에 최적화.

- **데이터셋**: [Emilia-YODAS Korean](https://huggingface.co/datasets/amphion/Emilia-Dataset) (7,300h, CC BY 4.0)
- **언어 혼합**: ko=80%, en=10%, ja=5%, zh=5% (catastrophic forgetting 방지)
- **평가 기준**: FLEURS Korean CER ≤ 7.12 (모델 카드 baseline)
- **실행 환경**: RunPod GPU Pod (RTX 6000 Ada / A6000 / A100)
- **체크포인트 백업**: `saya6k/nemotron-kor-checkpoints` (private) — spot instance 안전장치

---

### Notebook Cell Map (17 cells)

| # | Section | | # | Section |
|---|---------|---|----|--------|
| 1 | Header | | 10 | Tokenizer (MD) |
| 2 | Setup + RunPod + .nemo 자동다운로드 | | 11 | Tokenizer 검증 |
| 3 | GPU Benchmark | | 12 | Training 설정 |
| 4 | 데이터 준비 (MD) | | 13 | Fine-Tuning |
| 5 | Data Ingest | | 13b | ⚡ HF Hub 백업 |
| 6 | Manifest (MD) | | 14 | 평가 (MD) |
| 7 | Build Manifest | | 15 | Checkpoint Sweep |
| 8 | Language Mix | | 16 | 결과 요약 |
| 9 | TTS Augmentation (OFF) | | 17 | Cost Report + Auto-Shutdown |

> **참고**: [NVIDIA 공식 튜토리얼](https://github.com/nvidia-riva/tutorials) + [Discussion #11](https://huggingface.co/nvidia/nemotron-3.5-asr-streaming-0.6b/discussions/11)

In [ ]:
# ============================================================
# Cell 2: Environment Setup + RunPod Automation
# ============================================================
import os, sys, subprocess, logging, shlex, json
from pathlib import Path

# ---- Logging ----
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('asr_finetune_kor')

# ---- GPU Check ----
logger.info("GPU 확인 중...")
try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
         '--format=csv,noheader,nounits'],
        capture_output=True, text=True, check=True
    )
    for line in result.stdout.strip().split('\n'):
        parts = [p.strip() for p in line.split(',')]
        logger.info(f"  GPU: {parts[0]} | VRAM: {parts[1]}MB total, {parts[2]}MB free")
except (FileNotFoundError, subprocess.CalledProcessError) as e:
    raise RuntimeError(
        "GPU를 찾을 수 없습니다. CUDA 지원 GPU가 필요합니다."
    ) from e

# ---- Path Configuration (RunPod) ----
WORKSPACE = Path(os.environ.get('WORKSPACE', '/workspace'))
DATA_DIR = Path(os.environ.get('DATA_DIR', str(WORKSPACE / 'data')))
NEMO_DIR = Path(os.environ.get('NEMO_DIR', str(WORKSPACE / 'NeMo')))
BATCH_DURATION = int(os.environ.get('BATCH_DURATION', '200'))
MAX_EPOCHS = int(os.environ.get('MAX_EPOCHS', '3'))
GPU_HOURLY_RATE = float(os.environ.get('GPU_HOURLY_RATE', '0.79'))

# Create directories
for d in [DATA_DIR, DATA_DIR / 'raw', DATA_DIR / 'processed',
          DATA_DIR / 'wav_cache', DATA_DIR / 'checkpoints',
          DATA_DIR / 'results']:
    d.mkdir(parents=True, exist_ok=True)

logger.info(f"DATA_DIR:     {DATA_DIR}")
logger.info(f"MAX_EPOCHS:   {MAX_EPOCHS}")
logger.info(f"BATCH_DURATION: {BATCH_DURATION}")
logger.info(f"GPU rate:     ${GPU_HOURLY_RATE}/h")

# ---- Dependency Installation ----
!pip install -q wget text-unidecode 'matplotlib>=3.3.2' Cython
!pip install -q huggingface-hub==0.23.2 librosa datasets soundfile tqdm jiwer gTTS runpod
!apt-get install -y -qq sox libsndfile1 ffmpeg libsox-fmt-mp3 jq 2>/dev/null

# ---- NeMo Installation (main branch) ----
BRANCH = 'main'
!python -m pip install -q "nemo_toolkit[asr] @ git+https://github.com/NVIDIA/NeMo.git@{BRANCH}"

# ---- Clone NeMo source (needed for example scripts) ----
if not NEMO_DIR.exists():
    !git clone -b $BRANCH https://github.com/NVIDIA-NeMo/NeMo $NEMO_DIR
os.environ["PYTHONPATH"] = f"{NEMO_DIR}:{os.environ.get('PYTHONPATH', '')}"

# ---- Add scripts to path ----
SCRIPTS_DIR = Path.cwd() / 'scripts'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

# ---- Import torch to confirm CUDA ----
import torch
logger.info(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    logger.info(f"  Device: {torch.cuda.get_device_name(0)}")
    logger.info(f"  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

# ---- Base Model (.nemo) Resolution ----
# Priority: 1) HF_CKPT env var, 2) WORKSPACE에서 자동 검색, 3) HF Hub에서 다운로드
HF_CKPT = os.environ.get('HF_CKPT', '')

if not HF_CKPT or not os.path.isfile(HF_CKPT):
    # Auto-search workspace for .nemo files
    logger.info("HF_CKPT 미설정 — .nemo 파일 검색 중...")
    nemo_candidates = sorted(Path(str(WORKSPACE)).rglob('*.nemo'))
    
    if nemo_candidates:
        HF_CKPT = str(nemo_candidates[0])
        logger.info(f"  발견: {HF_CKPT}")
    else:
        # Download from HuggingFace Hub
        logger.info("  .nemo 파일 없음 — HuggingFace Hub에서 다운로드합니다...")
        logger.info("  (최초 1회만 다운로드, 이후 캐시 사용)")
        
        from huggingface_hub import snapshot_download
        MODEL_REPO = os.environ.get('HF_MODEL_REPO', 'nvidia/nemotron-3.5-asr-streaming-0.6b')
        
        try:
            downloaded = snapshot_download(
                repo_id=MODEL_REPO,
                local_dir=str(DATA_DIR / 'base_model'),
                local_dir_use_symlinks=False,
            )
            nemo_files = sorted(Path(downloaded).rglob('*.nemo'))
            if nemo_files:
                HF_CKPT = str(nemo_files[0])
                logger.info(f"  다운로드 완료: {HF_CKPT}")
            else:
                raise FileNotFoundError(f"No .nemo file found in {MODEL_REPO}")
        except Exception as e:
            raise RuntimeError(
                f"모델 다운로드 실패. 수동으로 설정하세요: export HF_CKPT=/path/to/model.nemo\n"
                f"  Error: {e}"
            ) from e

assert os.path.isfile(HF_CKPT), f".nemo 파일이 없습니다: {HF_CKPT}"
file_size_gb = os.path.getsize(HF_CKPT) / 1024**3
logger.info(f"HF_CKPT: {HF_CKPT} ({file_size_gb:.2f} GB)")

# ---- RunPod Detection & Spot Guard ----
from runpod_auto import is_runpod, register_spot_guard

ON_RUNPOD = is_runpod()
if ON_RUNPOD:
    pod_id = os.environ.get('RUNPOD_POD_ID', 'unknown')
    pod_type = os.environ.get('RUNPOD_POD_TYPE', 'on-demand')
    logger.info(f"🖥️  RunPod 감지: Pod {pod_id} ({pod_type})")
    
    # Register spot instance preemption handler
    # Saves checkpoint to HF Hub on SIGTERM (~30s before reclaim)
    register_spot_guard(checkpoint_dir=str(DATA_DIR / 'checkpoints'))
    
    # Auto-detect GPU hourly rate from RunPod if not set
    if not os.environ.get('GPU_HOURLY_RATE'):
        try:
            import runpod as rp
            rp.api_key = os.environ.get('RUNPOD_API_KEY', '')
            if rp.api_key:
                pod_info = rp.get_pod(pod_id)
                detected_rate = pod_info.get('costPerHr', 0)
                if detected_rate > 0:
                    GPU_HOURLY_RATE = detected_rate
                    os.environ['GPU_HOURLY_RATE'] = str(detected_rate)
                    logger.info(f"  GPU rate (auto): ${detected_rate:.2f}/h")
        except Exception:
            pass
else:
    logger.info("로컬 환경 (RunPod 아님) — spot guard 비활성화")

logger.info("환경 설정 완료.")

In [ ]:
# ============================================================
# Cell 3: GPU Benchmark — 속도/VRAM 측정 및 비용 추정
# ============================================================
"""
본 셀은 실제 학습 전에 GPU 성능을 측정하여:
1. steps/sec (학습 처리량)
2. Peak VRAM 사용량
3. OOM 없는 최대 batch_duration
4. 전체 7,300h × 3 epochs 예상 비용

을 추정합니다.
"""

# Step 1: Create a tiny benchmark manifest (200 samples from Emilia-YODAS KO)
logger.info("벤치마크용 매니페스트 생성 중...")

try:
    from datasets import load_dataset
    
    ds = load_dataset(
        "amphion/Emilia-Dataset",
        data_files={"train": "Emilia-YODAS/KO/**/*.tar"},
        split="train",
        streaming=True
    )
    
    # 빠른 MP3→WAV 변환, 200샘플만
    import soundfile as sf
    from datasets import Audio
    
    bench_dir = DATA_DIR / 'bench'
    bench_dir.mkdir(exist_ok=True)
    bench_manifest_path = str(bench_dir / 'bench_manifest.json')
    bench_samples = 0
    
    with open(bench_manifest_path, 'w', encoding='utf-8') as f:
        for sample in ds:
            if bench_samples >= 200:
                break
            text = sample.get('text', '').strip()
            if not text or len(text) < 5:
                continue
            
            audio = sample['audio']
            wav_path = str(bench_dir / f"bench_{bench_samples:04d}.wav")
            audio_array = audio['array']
            sr = audio['sampling_rate']
            
            # 16kHz 리샘플링
            if sr != 16000:
                import librosa
                audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=16000)
                sr = 16000
            
            sf.write(wav_path, audio_array, sr)
            duration = len(audio_array) / sr
            
            entry = {
                "audio_filepath": wav_path,
                "duration": round(duration, 3),
                "text": text,
                "lang": "ko-KR",
                "target_lang": "ko-KR"
            }
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')
            bench_samples += 1
    
    logger.info(f"벤치마크 매니페스트: {bench_samples} samples → {bench_manifest_path}")
    
    # Step 2: Run benchmark
    from benchmark_gpu import run_benchmark
    
    # HF_CKPT 확인
    assert HF_CKPT, "환경 변수 HF_CKPT가 설정되지 않았습니다. .nemo 모델 경로를 설정하세요."
    assert os.path.isfile(HF_CKPT), f"HF_CKPT 파일을 찾을 수 없습니다: {HF_CKPT}"
    
    bench_results = run_benchmark(
        nemo_dir=str(NEMO_DIR),
        hf_ckpt=HF_CKPT,
        bench_manifest=bench_manifest_path,
        batch_duration=BATCH_DURATION,
        max_steps=500,  # 500~1000 steps 권장
        output_dir=str(DATA_DIR / 'bench'),
    )
    
    if bench_results.get('oom_detected'):
        logger.error("OOM 발생! BATCH_DURATION을 낮춰서 다시 실행하세요.")
        logger.error(f"  export BATCH_DURATION={int(BATCH_DURATION * 0.6)}")
    
except ImportError as e:
    logger.warning(f"벤치마크를 실행할 수 없습니다 (필요 패키지 부족): {e}")
    logger.info("벤치마크 없이 진행합니다. 비용은 수동 추정하세요.")

---

## 데이터 준비

### Emilia-YODAS Korean (7,300h, CC BY 4.0)

Emilia-YODAS는 WebDataset tar 포맷으로 제공됩니다. 각 tar에는 MP3 오디오 + JSON 메타데이터가 포함되어 있습니다.

```json
{
  "id": "...",
  "text": "한국어 트랜스크립트",
  "duration": 3.5,
  "speaker": "...",
  "language": "ko",
  "dnsmos": 3.29
}
```

### 평가 데이터셋 6종

| # | 이름 | 용도 | 소스 |
|---|------|------|------|
| 1 | `val_emilia_holdout_ko` | **Validation** (Early Stopping) | Emilia-YODAS KO hold-out |
| 2 | `test_fleurs_ko` | Test | FLEURS ko_kr |
| 3 | `test_emilia_holdout_ko` | Test | Emilia-YODAS KO hold-out B (학습/val 미사용) |
| 4 | `test_zeroth_ko` | Test | Zeroth Korean |
| 5 | `test_mixed_en` | Test | Emilia-YODAS EN |
| 6 | `test_mixed_ja` | Test | Emilia-YODAS JA |

> **중요**: Validation과 Test는 분리되어 있습니다. Early Stopping은 validation만 사용하고,
> Test 셋은 checkpoint sweep 평가에만 사용됩니다 (test leakage 방지).

### 외부 평가 데이터셋 (수동 다운로드 필요)

| Dataset | Source | Download |
|---------|--------|----------|
| Emilia-YODAS KO holdout B | Emilia-YODAS | 자동 (Cell 7에서 분할) ✅ |
| Zeroth Korean | GitHub | `git clone https://github.com/goodatlas/zeroth` → `data/raw/zeroth/` |

> 두 데이터셋은 라이선스 문제로 자동 다운로드가 불가능합니다. 
> 디렉토리가 존재하면 Cell 7에서 자동으로 manifest를 생성합니다.

In [ ]:
# ============================================================
# Cell 5: Data Ingest — Emilia-YODAS Korean Streaming
# ============================================================
"""
HF datasets streaming으로 Emilia-YODAS Korean을 로드합니다.

설정 가능한 환경 변수:
  DNSMOS_THRESHOLD: DNSMOS 필터 임계값 (미설정 = 원본 유지)
  SMOKE_N: 스모크 서브셋 크기 (기본: 5000)
  HOLD_OUT_N: Validation hold-out 크기 (기본: 1000)
"""
import soundfile as sf
import librosa
from datasets import load_dataset
from tqdm import tqdm

DNSMOS_THRESHOLD = float(os.environ.get('DNSMOS_THRESHOLD', '0'))  # 0 = 비활성화
SMOKE_N = int(os.environ.get('SMOKE_N', '5000'))
HOLD_OUT_N = int(os.environ.get('HOLD_OUT_N', '1000'))

WAV_CACHE_DIR = DATA_DIR / 'wav_cache'
WAV_CACHE_DIR.mkdir(exist_ok=True)

# ---- Emilia-YODAS Korean 로드 (streaming) ----
logger.info("Emilia-YODAS Korean 로드 중 (streaming)...")
ds = load_dataset(
    "amphion/Emilia-Dataset",
    data_files={"train": "Emilia-YODAS/KO/**/*.tar"},
    split="train",
    streaming=True
)

# ---- 처리 루프 ----
all_entries = []
skipped_dnsmos = 0
skipped_text = 0
wav_cache_hits = 0

logger.info(f"DNSMOS 필터: {'≥ ' + str(DNSMOS_THRESHOLD) if DNSMOS_THRESHOLD > 0 else '비활성화 (원본 유지)'}")
logger.info(f"스모크 서브셋: {SMOKE_N}, Hold-out: {HOLD_OUT_N}")

for sample in tqdm(ds, desc="Streaming 처리"):
    text = sample.get('text', '').strip()
    if not text:
        skipped_text += 1
        continue
    
    # DNSMOS 필터 (옵션)
    dnsmos = sample.get('dnsmos', 0)
    if DNSMOS_THRESHOLD > 0 and dnsmos < DNSMOS_THRESHOLD:
        skipped_dnsmos += 1
        continue
    
    # MP3 → WAV 변환 (캐시 사용)
    audio = sample['audio']
    wav_filename = f"{sample.get('id', str(abs(hash(text))))}.wav"
    wav_path = WAV_CACHE_DIR / wav_filename
    
    if wav_path.exists():
        wav_cache_hits += 1
        duration = librosa.get_duration(path=str(wav_path))
    else:
        audio_array = audio['array']
        sr = audio['sampling_rate']
        if sr != 16000:
            audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=16000)
            sr = 16000
        sf.write(str(wav_path), audio_array, sr)
        duration = len(audio_array) / sr
    
    all_entries.append({
        "audio_filepath": str(wav_path),
        "duration": round(duration, 3),
        "text": text,
        "lang": "ko-KR",
        "target_lang": "ko-KR",
    })

    if len(all_entries) >= SMOKE_N:
        logger.info(f"SMOKE_N={SMOKE_N} 도달, 처리 중단")
        break

logger.info(f"처리 완료: {len(all_entries)} entries")
logger.info(f"  WAV 캐시 히트: {wav_cache_hits}")
logger.info(f"  DNSMOS 스킵: {skipped_dnsmos}")
logger.info(f"  빈 텍스트 스킵: {skipped_text}")

---

### NeMo Manifest Format

각 라인은 JSON 객체:
```json
{"audio_filepath": "/path/to/file.wav", "duration": 3.5, "text": "한국어 트랜스크립트", "lang": "ko-KR", "target_lang": "ko-KR"}
```

언어 혼합 시, 해당 언어의 `lang` 태그를 그대로 유지합니다 (예: `"en-US"`, `"ja-JP"`, `"zh-CN"`).

In [ ]:
# ============================================================
# Cell 7: Build Manifest — train/val/holdout + eval datasets
# ============================================================
import random
random.seed(42)

PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)

# ---- Shuffle & split ----
random.shuffle(all_entries)

# Hold-out: 처음 HOLD_OUT_N개 → validation용
holdout_entries = all_entries[:HOLD_OUT_N]
# 나머지는 학습/테스트로 사용
rest_entries = all_entries[HOLD_OUT_N:]

# 학습: SMOKE_N개 (스모크) 또는 전체 (본학습)
train_entries = rest_entries[:SMOKE_N]

# ---- Write manifests ----
def write_manifest(path, entries):
    with open(path, 'w', encoding='utf-8') as f:
        for e in entries:
            f.write(json.dumps(e, ensure_ascii=False) + '\n')
    logger.info(f"  {path}: {len(entries)} entries")

# Training manifest
TRAIN_MANIFEST = str(PROCESSED_DIR / 'train_manifest_ko.json')
write_manifest(TRAIN_MANIFEST, train_entries)

# Full manifest (for later full training, Track B)
FULL_TRAIN_MANIFEST = str(PROCESSED_DIR / 'train_manifest_full_ko.json')
write_manifest(FULL_TRAIN_MANIFEST, rest_entries)

# Validation manifest (Emilia holdout → Early Stopping)
VAL_MANIFEST = str(PROCESSED_DIR / 'val_emilia_holdout_ko.json')
write_manifest(VAL_MANIFEST, holdout_entries)

# ---- Validation: 모든 항목 ko-KR 태그 확인 ----
for path in [TRAIN_MANIFEST, VAL_MANIFEST]:
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            entry = json.loads(line)
            assert entry['target_lang'] == 'ko-KR', f"{path}:{i} target_lang={entry['target_lang']}"
            assert entry['text'].strip(), f"{path}:{i} text is empty"
            assert entry['duration'] > 0, f"{path}:{i} duration={entry['duration']}"
logger.info("Manifest 검증 통과: 모든 항목이 ko-KR 태그와 유효한 텍스트/duration을 가집니다.")

# ---- FLEURS Korean eval manifest (Test only) ----
logger.info("FLEURS Korean eval manifest 생성 중...")
try:
    fleurs_ds = load_dataset("google/fleurs", "ko_kr", split="test")
    fleurs_entries = []
    for sample in fleurs_ds:
        audio = sample['audio']
        wav_path = str(WAV_CACHE_DIR / f"fleurs_{sample['id']}.wav")
        audio_array = audio['array']
        sr = audio['sampling_rate']
        if sr != 16000:
            audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=16000)
        sf.write(wav_path, audio_array, 16000)
        fleurs_entries.append({
            "audio_filepath": wav_path,
            "duration": round(len(audio_array) / 16000, 3),
            "text": sample['transcription'].strip(),
            "lang": "ko-KR",
            "target_lang": "ko-KR"
        })
    FLEURS_MANIFEST = str(PROCESSED_DIR / 'test_fleurs_ko.json')
    write_manifest(FLEURS_MANIFEST, fleurs_entries)
except Exception as e:
    logger.warning(f"FLEURS eval manifest 생성 실패: {e}")
    FLEURS_MANIFEST = None

# ---- Emilia-YODAS KO test holdout (Test only) ----
# val_emilia_holdout(1000개)와 별도로 분리된 500개 테스트셋
# Early Stopping에 사용되지 않으므로 test leakage 없음
TEST_HOLD_OUT_N = 500
# rest_entries에서 이미 train_entries(SMOKE_N개)를 제외하고 남은 것에서 추출
# rest_entries = all_entries[HOLD_OUT_N:] → train_entries = rest_entries[:SMOKE_N]
# test holdout은 rest_entries의 뒤쪽에서 가져옴 (train과 겹치지 않음)
test_holdout_start = SMOKE_N
test_holdout_end = test_holdout_start + TEST_HOLD_OUT_N
test_holdout_entries = rest_entries[test_holdout_start:test_holdout_end]
TEST_EMILIA_MANIFEST = str(PROCESSED_DIR / 'test_emilia_holdout_ko.json')
write_manifest(TEST_EMILIA_MANIFEST, test_holdout_entries)
logger.info(f"  Emilia holdout test: {len(test_holdout_entries)} entries (독립적, 학습/val 미사용)")

# ---- Zeroth Korean eval manifest (Test only) ----
# 다운로드: git clone https://github.com/goodatlas/zeroth.git → data/raw/zeroth/
ZEROTH_DIR = DATA_DIR / 'raw' / 'zeroth'
ZEROTH_MANIFEST = None
if ZEROTH_DIR.exists():
    logger.info("Zeroth Korean eval manifest 생성 중...")
    try:
        zeroth_test = ZEROTH_DIR / 'data' / 'test'
        zeroth_text = zeroth_test / 'text'
        # Zeroth uses Kaldi wav.scp — find audio files via scp
        zeroth_wav_scp = zeroth_test / 'wav.scp'
        zeroth_audio_dir = zeroth_test / 'wav'
        if zeroth_text.exists():
            from build_manifest import build_manifest
            zeroth_output = str(PROCESSED_DIR / 'test_zeroth_ko.json')
            n = build_manifest(
                audio_dir=str(zeroth_audio_dir) if zeroth_audio_dir.exists() else str(zeroth_test),
                output_path=zeroth_output,
                transcript_file=str(zeroth_text),
                transcript_format='kaldi',
                lang='ko-KR',
                target_lang='ko-KR',
                audio_ext='.flac',
                skip_missing_transcripts=True,
            )
            if n > 0:
                ZEROTH_MANIFEST = zeroth_output
                logger.info(f"  Zeroth Korean: {n} entries")
    except Exception as e:
        logger.warning(f"Zeroth Korean manifest 생성 실패: {e}")
else:
    logger.info("Zeroth Korean: 다운로드 필요 — https://github.com/goodatlas/zeroth")

In [ ]:
# ============================================================
# Cell 8: Language Mix — ko=80%, en=10%, ja=5%, zh=5%
# ============================================================
"""
Catastrophic forgetting 방지를 위해 학습 데이터에 타언어를 혼합합니다.
모두 같은 Emilia-YODAS 데이터셋에서 추출하여 포맷 일관성을 유지합니다.
"""

LANG_MIX_RATIO = {
    "ko": float(os.environ.get('LANG_MIX_KO', '0.80')),
    "en": float(os.environ.get('LANG_MIX_EN', '0.10')),
    "ja": float(os.environ.get('LANG_MIX_JA', '0.05')),
    "zh": float(os.environ.get('LANG_MIX_ZH', '0.05')),
}

# 비율 합계 검증
total_ratio = sum(LANG_MIX_RATIO.values())
assert abs(total_ratio - 1.0) < 0.01, f"언어 혼합 비율 합계 = {total_ratio}, 1.0이어야 함"

OTHER_LANGS = [
    ("en", "Emilia-YODAS/EN/**/*.tar", "en-US"),
    ("ja", "Emilia-YODAS/JA/**/*.tar", "ja-JP"),
    ("zh", "Emilia-YODAS/ZH/**/*.tar", "zh-CN"),
]

# ---- 기본 학습 매니페스트 로드 ----
with open(TRAIN_MANIFEST, 'r', encoding='utf-8') as f:
    ko_entries = [json.loads(line) for line in f]
n_ko = len(ko_entries)

# 목표: ko_entries가 LANG_MIX_RATIO["ko"] 비율을 차지하도록
# total = n_ko / LANG_MIX_RATIO["ko"]
# other_total = total - n_ko
total_entries = int(n_ko / LANG_MIX_RATIO["ko"])
other_total = total_entries - n_ko

logger.info(f"한국어: {n_ko} entries ({LANG_MIX_RATIO['ko']:.0%})")
logger.info(f"타언어 합계: {other_total} entries ({1 - LANG_MIX_RATIO['ko']:.0%})")

# ---- 타언어 로드 ----
mixed_entries = list(ko_entries)
eval_other_manifests = {}

for lang_code, data_files_pattern, lang_tag in OTHER_LANGS:
    target_n = int(other_total * LANG_MIX_RATIO[lang_code] / (1 - LANG_MIX_RATIO["ko"]))
    logger.info(f"  {lang_code} 로드 중 (목표: {target_n})...")
    
    try:
        other_ds = load_dataset(
            "amphion/Emilia-Dataset",
            data_files={"train": data_files_pattern},
            split="train",
            streaming=True
        )
        
        lang_entries = []
        for sample in other_ds:
            if len(lang_entries) >= target_n + 500:  # 500 extra for eval
                break
            text = sample.get('text', '').strip()
            if not text:
                continue
            
            audio = sample['audio']
            wav_filename = f"{lang_code}_{sample.get('id', str(abs(hash(text))))}.wav"
            wav_path = WAV_CACHE_DIR / wav_filename
            
            if not wav_path.exists():
                audio_array = audio['array']
                sr = audio['sampling_rate']
                if sr != 16000:
                    audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=16000)
                sf.write(str(wav_path), audio_array, 16000)
            duration = librosa.get_duration(path=str(wav_path))
            
            lang_entries.append({
                "audio_filepath": str(wav_path),
                "duration": round(duration, 3),
                "text": text,
                "lang": lang_tag,
                "target_lang": lang_tag,
            })
        
        # 상위 target_n개 → 학습, 나머지 500개 → eval
        mixed_entries.extend(lang_entries[:target_n])
        if len(lang_entries) > target_n:
            eval_entries = lang_entries[target_n:target_n + 500]
            eval_path = str(PROCESSED_DIR / f'test_mixed_{lang_code}.json')
            write_manifest(eval_path, eval_entries)
            eval_other_manifests[f"mixed_{lang_code}"] = eval_path
        
        logger.info(f"    {len(lang_entries[:target_n])} entries + {min(500, max(0, len(lang_entries) - target_n))} eval")
    except Exception as e:
        logger.warning(f"  {lang_code} 로드 실패: {e}")

# ---- Shuffle & write mixed manifest ----
random.shuffle(mixed_entries)
MIXED_MANIFEST = str(PROCESSED_DIR / 'train_manifest_mixed.json')
write_manifest(MIXED_MANIFEST, mixed_entries)

# ---- 비율 검증 ----
ko_count = sum(1 for e in mixed_entries if e.get('lang') == 'ko-KR' or e.get('target_lang') == 'ko-KR')
actual_ko_ratio = ko_count / len(mixed_entries)
logger.info(f"혼합 완료: {len(mixed_entries)} entries, 한국어 비율: {actual_ko_ratio:.1%}")
assert 0.70 <= actual_ko_ratio <= 0.85, \
    f"한국어 비율 {actual_ko_ratio:.1%}이 허용 범위 [70%, 85%]를 벗어났습니다"
logger.info("언어 혼합 비율 검증 통과.")

In [ ]:
# ============================================================
# Cell 9: TTS Augmentation (선택 — 기본 OFF)
# ============================================================
"""
gTTS 기반 희귀 도메인 용어 합성.

7,300h 실제 데이터가 있으므로 대부분의 경우 불필요.
TTS_AUGMENT=true 로 설정해야만 실행됩니다.
"""

TTS_AUGMENT = os.environ.get('TTS_AUGMENT', 'false').lower() == 'true'

if not TTS_AUGMENT:
    logger.info("TTS Augmentation: 비활성화 (기본).")
    logger.info("  활성화: export TTS_AUGMENT=true")
else:
    try:
        from tts_augment import generate_tts_samples
        
        # 사용자 정의 도메인 용어 목록 (필요시 수정)
        DOMAIN_TERMS = os.environ.get('TTS_TERMS', '').split(',')
        DOMAIN_TERMS = [t.strip() for t in DOMAIN_TERMS if t.strip()]
        
        if not DOMAIN_TERMS:
            logger.warning("TTS_AUGMENT=true 이지만 TTS_TERMS가 비어 있습니다.")
            logger.info("  예: export TTS_TERMS='반도체,인공지능,클라우드 컴퓨팅'")
        else:
            TTS_OUTPUT_DIR = DATA_DIR / 'tts_augmented'
            tts_entries = generate_tts_samples(
                term_list=DOMAIN_TERMS,
                output_dir=str(TTS_OUTPUT_DIR),
                lang="ko",
                engine="gtts",
            )
            
            if tts_entries:
                # 혼합 매니페스트에 병합
                with open(MIXED_MANIFEST, 'r', encoding='utf-8') as f:
                    existing = [json.loads(line) for line in f]
                combined = existing + tts_entries
                random.shuffle(combined)
                with open(MIXED_MANIFEST, 'w', encoding='utf-8') as f:
                    for e in combined:
                        f.write(json.dumps(e, ensure_ascii=False) + '\n')
                logger.info(f"TTS {len(tts_entries)}개 → 혼합 매니페스트 병합 완료.")
    except ImportError as e:
        logger.warning(f"TTS augmentation 불가: {e}")

---

## 토크나이저

데이터가 50시간 미만일 때는 pretrained 모델의 토크나이저를 재사용하는 것이 권장됩니다.
본 프로젝트는 7,300h 데이터를 사용하지만, 우선 pretrained 토크나이저를 검증한 후 필요시 adaptation을 검토합니다.

### 검증 기준
- **coverage ≥ 98%**: 한국어 텍스트가 정상적으로 토크나이징되는 비율
- **byte fallback ≤ 2%**: 토크나이저가 바이트 단위로 분해하는 비율
- **UNK ≈ 0%**: unknown token 비율
- **음절 분해**: 한글 음절이 자모 단위로 분해되지 않는지 확인

In [ ]:
# ============================================================
# Cell 11: Tokenizer 검증 — byte fallback, UNK, coverage
# ============================================================
import tarfile
import nemo.collections.asr as nemo_asr

# ---- .nemo 체크포인트 확인 ----
assert HF_CKPT, "HF_CKPT가 설정되지 않았습니다."
assert os.path.isfile(HF_CKPT), f".nemo 파일 없음: {HF_CKPT}"

logger.info(f"토크나이저 로드: {HF_CKPT}")

# ---- 토크나이저 추출 ----
with tarfile.open(HF_CKPT, 'r') as tar:
    tokenizer_files = [n for n in tar.getnames() if 'token' in n.lower()]
    logger.info(f"토크나이저 관련 파일: {tokenizer_files}")

# ---- Tokenizer 로드 ----
model = nemo_asr.models.EncDecRNNTBPEModel.from_pretrained(HF_CKPT)
tokenizer = model.tokenizer

# ---- 한국어 샘플 검증 ----
KOREAN_TEST_TEXTS = [
    "안녕하세요, 음성 인식 모델입니다.",
    "한국어 데이터로 파인튜닝을 수행합니다.",
    "네모 트로나이트 모델을 사용합니다.",
    "인공지능과 반도체는 대한민국의 핵심 산업입니다.",
    "오늘 날씨가 매우 좋네요.",
]

logger.info("\n토크나이저 한국어 커버리지 테스트:")
total_orig = 0
total_decoded = 0

for text in KOREAN_TEST_TEXTS:
    tokens = tokenizer.text_to_ids(text)
    decoded = tokenizer.ids_to_text(tokens)
    
    orig_chars = len(text.replace(' ', ''))
    decoded_chars = len(decoded.replace(' ', ''))
    total_orig += orig_chars
    total_decoded += decoded_chars
    
    logger.info(f"  원본: {text[:50]}")
    logger.info(f"  복원: {decoded[:50]}")
    logger.info(f"  토큰 수: {len(tokens)}, 문자: {orig_chars}→{decoded_chars}")

coverage = total_decoded / max(total_orig, 1)
logger.info(f"\n커버리지: {coverage:.1%}")

# ---- 음절 분해 검사 ----
sample_text = "안녕하세요"
tokens = tokenizer.text_to_ids(sample_text)
decoded = tokenizer.ids_to_text(tokens)

# 한글 완성형 음절이 자모로 분해되었는지 확인
import unicodedata
has_jamo = any('ᅟ' <= c <= 'ᇿ' or 'ᄀ' <= c <= 'ᇿ' for c in unicodedata.normalize('NFD', decoded))
if has_jamo:
    logger.warning("⚠️  음절 분해 감지: 한글 음절이 자모 단위로 분해되고 있습니다.")
    logger.warning("  토크나이저 adaptation을 검토하세요.")
else:
    logger.info("음절 분해: 없음 (정상)")

# ---- 최종 판정 ----
if coverage >= 0.98:
    logger.info(f"✅ 토크나이저 커버리지 양호: {coverage:.1%} ≥ 98%")
elif coverage >= 0.80:
    logger.warning(f"⚠️  토크나이저 커버리지 부족: {coverage:.1%} < 98%")
    logger.warning("  NeMo의 process_asr_text_tokenizer.py로 tokenizer adaptation을 검토하세요.")
else:
    logger.error(f"❌ 토크나이저 커버리지 심각: {coverage:.1%} < 80%")
    logger.error("  토크나이저 재학습이 필요합니다.")

# ---- byte fallback / UNK 비율 추정 ----
# BPE 토크나이저는 일반적으로 byte-level fallback을 사용함
logger.info("\n참고: BPE 토크나이저는 byte-level fallback이 기본 동작입니다.")
logger.info("  byte fallback ≤ 2%, UNK ≈ 0%를 목표로 하세요.")

---

## 학습 설정

### 핵심 하이퍼파라미터

| Parameter | Demo (AN4) | Korean Production | Reason |
|-----------|-----------|-------------------|--------|
| `lr` | 0.1 | **1e-4** | Pretrained representation 보존 |
| `max_epochs` | 1 | **3** | 7,300h × 3 = 충분한 노출 |
| `precision` | bf16 | bf16 | 메모리 효율 |
| `batch_duration` | 200 | 200 (OOM 시 감소) | GPU VRAM 의존 |
| `gradient_clip_val` | (default) | **1.0** | 학습 안정성 |
| `warmup` | 100 steps | **5%** (warmup_ratio) | Cosine decay 연계 |
| `limit_train_batches` | 60 | **제거** | NeMo #15782 |
| `seed` | (none) | **42** | 재현성 |

### Early Stopping
- Monitor: `val_cer` (Emilia holdout)
- Patience: 5 evaluations (5000 step 간격 → 25,000 steps)
- Mode: min

### Checkpoint Strategy
- `every_n_train_steps=5000` — 중간 평가용
- `save_top_k=3` — val_cer 기준 best 3개만 유지 (폭증 방지)
- `resume_if_exists=true` — RunPod spot instance 대응

### EMA (선택)
- NeMo 지원 확인 후 A/B 비교. Streaming transducer에서 효과 미미할 수 있음.

In [ ]:
# ============================================================
# Cell 13: Fine-Tuning — Production Korean ASR
# ============================================================
"""
학습 설정:
  - Validataion: val_emilia_holdout_ko (Early Stopping)
  - Training:   train_manifest_mixed.json (ko=80%, en=10%, ja=5%, zh=5%)
  - lr=1e-4, cosine decay, warmup_ratio=0.05, gradient_clip_val=1.0
  - seed_everything=42
  - max_epochs=3 (max_steps와 중복 사용 금지)
"""
import shlex

# ---- Validate prerequisites ----
assert os.path.isfile(HF_CKPT), f"Base .nemo not found: {HF_CKPT}"

# Use mixed manifest if exists, otherwise Korean-only
if os.path.exists(MIXED_MANIFEST):
    TRAIN_MANIFEST_PATH = MIXED_MANIFEST
    logger.info(f"학습 매니페스트: 혼합 (ko+en+ja+zh) → {TRAIN_MANIFEST_PATH}")
else:
    TRAIN_MANIFEST_PATH = TRAIN_MANIFEST
    logger.info(f"학습 매니페스트: 한국어 only → {TRAIN_MANIFEST_PATH}")

CHECKPOINT_DIR = str(DATA_DIR / 'checkpoints')

# ---- Training Command ----
# ⚠️  limit_train_batches: 절대 설정하지 않음 (NeMo Issue #15782)
# ⚠️  max_steps: max_epochs=3과 중복 사용 금지 (먼저 도달한 조건으로 종료)

! python {NEMO_DIR}/examples/asr/speech_to_text_finetune.py \
    --config-path="../asr/conf/fastconformer/cache_aware_streaming" \
    --config-name=fastconformer_transducer_bpe_streaming_prompt.yaml \
    +init_from_nemo_model={shlex.quote(HF_CKPT)} \
    ++model.train_ds.manifest_filepath={shlex.quote(TRAIN_MANIFEST_PATH)} \
    ++model.validation_ds.manifest_filepath={shlex.quote(VAL_MANIFEST)} \
    ++trainer.devices=1 \
    ++trainer.max_epochs={MAX_EPOCHS} \
    ++trainer.precision=bf16 \
    ++trainer.gradient_clip_val=1.0 \
    ++seed_everything=42 \
    ++model.train_ds.batch_duration={BATCH_DURATION} \
    ++model.optim.name="adamw" \
    ++model.optim.lr=0.0001 \
    ++model.optim.weight_decay=0.001 \
    ++model.optim.sched.name="CosineAnnealing" \
    ++model.optim.sched.warmup_ratio=0.05 \
    ++model.optim.sched.min_lr=1e-6 \
    ++exp_manager.exp_dir={shlex.quote(CHECKPOINT_DIR)} \
    ++exp_manager.resume_if_exists=true \
    ++exp_manager.checkpoint_every_n_train_steps=5000 \
    ++exp_manager.save_top_k=3 \
    ++exp_manager.early_stopping_enabled=true \
    ++exp_manager.early_stopping_metric="val_cer" \
    ++exp_manager.early_stopping_patience=5

logger.info("Fine-tuning 완료.")
logger.info(f"체크포인트: {CHECKPOINT_DIR}")

In [ ]:
# ============================================================
# Cell 13b: Checkpoint Backup → HF Hub
# ============================================================
"""
학습 직후 최종 체크포인트를 HF Hub에 백업합니다.
기본 repo: saya6k/nemotron-kor-checkpoints (private, auto-create)

환경 변수 (선택):
  HF_REPO_ID:  다른 repo 사용 시 (기본값: saya6k/nemotron-kor-checkpoints)
  HF_TOKEN:    HF API token (쓰기 권한 필요)
"""
from runpod_auto import upload_checkpoint_to_hub

DEFAULT_REPO = os.environ.get('HF_REPO_ID', 'saya6k/nemotron-kor-checkpoints')

# Find the newest checkpoint
ckpt_dir = Path(CHECKPOINT_DIR)
nemo_files = sorted(ckpt_dir.rglob('*.nemo'), key=lambda p: p.stat().st_mtime)

if nemo_files:
    latest_ckpt = str(nemo_files[-1])
    file_size_mb = os.path.getsize(latest_ckpt) / 1024**2
    logger.info(f"체크포인트 백업: {latest_ckpt} ({file_size_mb:.0f} MB) → {DEFAULT_REPO}")

    url = upload_checkpoint_to_hub(
        checkpoint_path=latest_ckpt,
        repo_id=DEFAULT_REPO,
        token=os.environ.get('HF_TOKEN'),
    )
    if url:
        logger.info(f"✅ 백업 완료: {url}")
    else:
        logger.warning("⚠️  백업 실패 — HF_TOKEN을 확인하세요")
else:
    logger.warning("백업할 .nemo 체크포인트를 찾을 수 없습니다.")

---

## 평가

### Checkpoint Sweep Evaluation

모든 `.nemo` 체크포인트 × 6종 eval dataset에 대해 CER/WER/SER를 측정합니다.

- **Validation** (학습 중 Early Stopping): `val_emilia_holdout_ko`
- **Test** (Checkpoint Sweep 전용): FLEURS, Emilia holdout, Zeroth, Mixed EN, Mixed JA

### Metrics
- **CER** (Character Error Rate): 한국어 주 메트릭 (모델 카드 기준: 7.12)
- **WER** (Word Error Rate): 숫자/영어혼입/고유명사 오류 포착
- **SER** (Sentence Error Rate): 문장 단위 오류율

In [ ]:
# ============================================================
# Cell 15: Checkpoint Sweep — 모든 체크포인트 × 6종 eval
# ============================================================
"""
eval_checkpoints.py를 사용하여 모든 .nemo 체크포인트에 대해
6종 eval dataset의 CER/WER/SER를 일괄 측정합니다.
"""
import json

# ---- Eval dataset 구성 ----
EVAL_DATASETS = {
    "val_emilia_holdout_ko": VAL_MANIFEST,
}

# Test datasets (존재하는 것만 추가)
if FLEURS_MANIFEST and os.path.exists(FLEURS_MANIFEST):
    EVAL_DATASETS["test_fleurs_ko"] = FLEURS_MANIFEST

for key, path in eval_other_manifests.items():
    if os.path.exists(path):
        EVAL_DATASETS[f"test_{key}"] = path

# 추가 eval 셋이 있다면 여기에 등록
if TEST_EMILIA_MANIFEST and os.path.exists(TEST_EMILIA_MANIFEST):
    EVAL_DATASETS["test_emilia_holdout_ko"] = TEST_EMILIA_MANIFEST
if ZEROTH_MANIFEST and os.path.exists(ZEROTH_MANIFEST):
    EVAL_DATASETS["test_zeroth_ko"] = ZEROTH_MANIFEST

logger.info(f"Eval datasets ({len(EVAL_DATASETS)}종): {list(EVAL_DATASETS.keys())}")

# ---- Run checkpoint sweep ----
RESULTS_DIR = DATA_DIR / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
RESULTS_CSV = str(RESULTS_DIR / 'checkpoint_eval.csv')

datasets_json = json.dumps(EVAL_DATASETS)

! python scripts/eval_checkpoints.py \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --datasets '{datasets_json}' \
    --output {RESULTS_CSV} \
    --nemo-dir {NEMO_DIR} \
    --target-lang ko-KR

logger.info(f"Checkpoint sweep 완료 → {RESULTS_CSV}")

In [ ]:
# ============================================================
# Cell 17: Cost Report + Auto-Shutdown
# ============================================================
"""
학습 완료 후 실제 소요 시간과 비용을 벤치마크 예상치와 비교.
RunPod 환경이면 예산 초과 시 자동 종료 옵션을 제공합니다.

환경 변수:
  AUTO_SHUTDOWN:   "true" → 학습 완료 후 Pod 자동 종료
  MAX_COST:        최대 예산($), 초과 시 경고 + 선택적 종료
  RUNPOD_API_KEY:  Pod 종료에 필요
"""
import json as json_mod
from runpod_auto import pod_status, stop_pod, is_runpod

BENCH_RESULTS_PATH = DATA_DIR / 'bench' / 'benchmark_results.json'
AUTO_SHUTDOWN = os.environ.get('AUTO_SHUTDOWN', 'false').lower() == 'true'
MAX_COST = float(os.environ.get('MAX_COST', '0'))

print("\n" + "=" * 60)
print("  Cost Report")
print("=" * 60)

# ---- Benchmark comparison ----
if BENCH_RESULTS_PATH.exists():
    with open(BENCH_RESULTS_PATH, 'r') as f:
        bench = json_mod.load(f)
    
    est_cost = bench.get('estimated_cost', 0)
    
    print(f"\n  [Benchmark 예상]")
    print(f"    Steps/sec:       {bench.get('steps_per_sec', 'N/A')}")
    print(f"    Peak VRAM:       {bench.get('peak_vram_mb', 'N/A')} MB")
    print(f"    예상 시간:        {bench.get('estimated_hours', 'N/A')}h")
    print(f"    예상 비용:        ${est_cost:.2f}")
    print(f"    GPU rate:        ${bench.get('gpu_hourly_rate', 'N/A')}/h")

# ---- RunPod real data ----
print(f"\n  [RunPod 실측]")

if is_runpod():
    pod_id = os.environ.get('RUNPOD_POD_ID', '')
    gpu_type_env = os.environ.get('RUNPOD_GPU_TYPE', 'N/A')
    print(f"    Pod ID:          {pod_id}")
    print(f"    GPU Type:        {gpu_type_env}")
    
    # Try to get actual cost from RunPod API
    try:
        status = pod_status(pod_id)
        print(f"    Pod 상태:        {status['status']}")
        print(f"    가동 시간:        {status['uptime_hours']:.1f}h")
        print(f"    현재 요금:        ${status['estimated_cost']:.4f}")
        print(f"    시간당 요금:      ${status['cost_per_hour']:.2f}/h")
        print(f"    GPU 사용률:       {status['gpu_utilization_pct']}%")
        
        actual_cost = status['estimated_cost']
        actual_hours = status['uptime_hours']
    except Exception as e:
        logger.warning(f"RunPod API 조회 실패: {e}")
        actual_cost = 0
        actual_hours = 0
else:
    print(f"    로컬 환경 — RunPod 비용 추적 불가")
    actual_cost = 0
    actual_hours = 0

# ---- Budget check ----
if MAX_COST > 0 and actual_cost > 0:
    print(f"\n  [예산 체크]")
    print(f"    최대 예산:       ${MAX_COST:.2f}")
    print(f"    현재 비용:       ${actual_cost:.2f}")
    if actual_cost > MAX_COST:
        print(f"    ⚠️  예산 초과! ${actual_cost - MAX_COST:.2f} over budget")
    else:
        print(f"    ✅ 예산 이내 (${MAX_COST - actual_cost:.2f} remaining)")

# ---- Auto-shutdown ----
print(f"\n  [자동 종료]")

if AUTO_SHUTDOWN and is_runpod():
    pod_id = os.environ.get('RUNPOD_POD_ID', '')
    logger.warning(f"⚠️  AUTO_SHUTDOWN 활성화 — 30초 후 Pod 종료")
    logger.info("취소: 셀 실행을 중단하거나 Ctrl+C")
    import time
    for i in range(30, 0, -5):
        print(f"  종료까지 {i}초...")
        time.sleep(5)
    
    try:
        stop_pod(pod_id)
        print("  ✅ Pod 종료 완료.")
    except Exception as e:
        logger.error(f"Pod 종료 실패: {e}")
        print(f"  수동 종료: python scripts/runpod_auto.py stop --pod-id {pod_id}")
else:
    if is_runpod():
        print(f"  비활성화. 활성화: export AUTO_SHUTDOWN=true")
        print(f"  수동 종료: python scripts/runpod_auto.py stop")
    else:
        print(f"  로컬 환경 — 해당 없음")

# ---- Recommendation ----
print(f"\n  [다음 학습 인스턴스 추천]")
print(f"    가성비 최적:     RTX 6000 Ada ($0.79/h, 48GB)")
print(f"    최고 속도:       A100 SXM ($1.99/h, 80GB)")
print(f"    예산 최소:       RTX 4090 ($0.44/h, 24GB) — batch_duration=120 권장")

print("=" * 60)